# CreditFlow - EDA

Synthetic proxy data. EDA only: no .fit(), no training, no GPU.

In [1]:
import os
import sys
import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings('ignore')
for p in [os.getcwd(), os.path.join(os.getcwd(), '..'), os.path.join(os.getcwd(), '..', '..')]:
    p = os.path.abspath(p)
    if os.path.isdir(os.path.join(p, 'pipeline')):
        sys.path.insert(0, p)
        break
RANDOM_STATE=42
N_ROWS=1000
np.random.seed(RANDOM_STATE)
print('config-ok')

config-ok


In [2]:
n=N_ROWS
income=np.random.randint(1500,15001,size=n)
age=np.random.randint(18,101,size=n)
employment_years=np.random.uniform(0,1,size=n)*(age-18)
loan_amount=np.random.randint(5000,50001,size=n)
loan_term=np.random.randint(12,61,size=n)
existing_debt=np.random.randint(0,5001,size=n)
credit_history=np.random.uniform(0,1,size=n)*(age-18)
previous_defaults=np.random.poisson(0.3,size=n)
default=np.random.binomial(1,12/100,size=n)
df=pd.DataFrame({'income':income,'age':age,'employment_years':employment_years,'loan_amount':loan_amount,'loan_term':loan_term,'existing_debt':existing_debt,'credit_history':credit_history,'previous_defaults':previous_defaults,'default':default})
print('shape',df.shape)

shape (1000, 9)


In [3]:
print('Q1 default vs non-default mean')
g=df.groupby('default').mean(numeric_only=True)
print(g.to_string())

Q1 default vs non-default mean
              income        age  employment_years   loan_amount  loan_term  existing_debt  credit_history  previous_defaults
default                                                                                                                     
0        8307.027304  58.137656         19.537488  27582.311718  36.524460    2544.170648       19.712543           0.312856
1        8123.793388  56.867769         21.729207  26591.628099  37.578512    2598.479339       18.654027           0.305785


In [4]:
print('Q3 imbalance')
vc=df['default'].value_counts()
print(vc.to_string())
print('P(default)',round(df['default'].mean(),4))

Q3 imbalance
default
0    879
1    121
P(default) 0.121


In [5]:
print('Q2 feature correlation with target')
corr=df.corr(numeric_only=True)
print(corr['default'].drop('default').sort_values(ascending=False).to_string())
print('P(default)',round(df['default'].mean(),4))

Q2 feature correlation with target
employment_years     0.040193
loan_term            0.024404
existing_debt        0.012058
previous_defaults   -0.004234
income              -0.015716
age                 -0.017404
credit_history      -0.019842
loan_amount         -0.025414
P(default) 0.121


In [6]:
print('Q4 leakage suspects')
pd_cross=pd.crosstab(df['previous_defaults'],df['default'])
print(pd_cross.head(8).to_string())
print('P(default) by previous_defaults:')
print(df.groupby('previous_defaults')['default'].mean().to_string())

Q4 leakage suspects
default              0   1
previous_defaults         
0                  641  87
1                  202  31
2                   35   3
3                    1   0
P(default) by previous_defaults:
previous_defaults
0    0.119505
1    0.133047
2    0.078947
3    0.000000


In [7]:
print('Q5 missing / outliers / duplicates')
from pipeline.validation.schemas import validate_dataframe
_,violations=validate_dataframe(df)
if not violations:
    print('clean: NO violations')
else:
    for v in violations:
        print(' -',v)

Q5 missing / outliers / duplicates
clean: NO violations


In [8]:
print('Q6 feature engineering demo')
from pipeline.feature_engineering.features import add_derived_features
feat,flags=add_derived_features(df.head(5).drop(columns=['default']))
print(feat.to_string())
print('flags:',flags if flags else 'none')

Q6 feature engineering demo
   income  age  employment_years  loan_amount  loan_term  existing_debt  credit_history  previous_defaults  debt_to_income  loan_to_income  debt_to_loan  employment_stability  credit_history_year_ratio
0    8770   51         24.509315        10889         54           4462       26.735316                  0        0.508780        1.241619      0.409771              1.000000                   0.810161
1    2360   23          2.127467        42822         50           2959        2.148367                  0        1.253814       18.144915      0.069100              0.212747                   0.429673
2    6890   70         17.988620        41636         38            183       38.992716                  0        0.026560        6.042961      0.004395              1.000000                   0.749860
3   14918   83         24.117520        25584         40           3358       59.347695                  0        0.225097        1.714975      0.131254            

In [9]:
print('Q7 validation demo: clean vs bad row')
from pipeline.validation.schemas import validate_dataframe
import pandas as pd
bad=pd.DataFrame({'income':[0],'age':[17],'employment_years':[5],'loan_amount':[10000],'loan_term':[36],'existing_debt':[3500],'credit_history':[5],'previous_defaults':[0]})
_,bad_v=validate_dataframe(bad)
for v in bad_v:
    print(' -',v)

Q7 validation demo: clean vs bad row
 - income: must be > 0
 - age: must be in [18,100]
 - employment_years: must be <= age - 18
 - credit_history: must be <= age - 18


In [10]:
print('Requirements check')
import sys
print('pandas',pd.__version__)
print('numpy',np.__version__)
forbidden=[m for m in ['xgboost','torch','tensorflow','sklearn'] if m in sys.modules]
print('forbidden modules loaded:',forbidden if forbidden else 'none - OK')
print('no .fit() in this notebook (EDA only)')

Requirements check
pandas 2.3.3
numpy 2.5.3
forbidden modules loaded: none - OK
no .fit() in this notebook (EDA only)
